# 🔄 Reverse Logistics Optimization — MILP via IBM CPLEX
### Thesis: Predictive Waste Classification and Reverse Logistics Optimization Using Machine Learning in a Circular Supply Chain
**Case Study: Thread Dyeing Company**  
**Solver: IBM ILOG CPLEX via DOcplex**  
**Kernel: `thesis_cplex` (Python 3.10)**

---
> ✅ **Before running:** Make sure you selected the `thesis_cplex` kernel  
> (Top menu → Kernel → Change Kernel → thesis_cplex)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Environment & Library Check
# Run this first to confirm everything is installed correctly
# ═══════════════════════════════════════════════════════════

import sys
print(f"Python version : {sys.version}")
print(f"Python path    : {sys.executable}")
print()

# Check docplex
try:
    import docplex
    print(f"✅ docplex      : {docplex.__version__}")
except ImportError:
    print("❌ docplex NOT found — run: pip install docplex")

# Check CPLEX engine
try:
    import cplex
    print(f"✅ cplex engine : {cplex.__version__}")
except ImportError:
    print("⚠️  cplex engine not found as standalone package")
    print("   (This is OK if IBM ILOG Studio is installed — DOcplex will find it automatically)")

# Verify DOcplex can see CPLEX
print()
print("─── DOcplex Environment Report ───")
import subprocess
result = subprocess.run([sys.executable, "-m", "docplex.mp.environment"], 
                       capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# Check other required packages
import importlib
for pkg in ["pandas", "numpy"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"✅ {pkg:<10}: {mod.__version__}")
    except ImportError:
        print(f"❌ {pkg} NOT found — run: pip install {pkg}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Load the MILP Model from reverse_logistics_cplex.py
# ═══════════════════════════════════════════════════════════

import sys
import os

# ┌─────────────────────────────────────────────────────────┐
# │  ⚠️  UPDATE THIS PATH to wherever you saved the .py file │
# └─────────────────────────────────────────────────────────┘
PY_FILE_FOLDER = r"C:\4-2\Thesis\data\Reverse logistics"
# Example: r"C:\Users\Rahim\Desktop\Thesis\MILP"
# Tip: In File Explorer, hold Shift + Right-click the folder → "Copy as path"

# Add folder to Python path so we can import from it
if PY_FILE_FOLDER not in sys.path:
    sys.path.insert(0, PY_FILE_FOLDER)

# Import all functions from the MILP script
from reverse_logistics_cplex import (
    build_and_solve_cplex,
    build_and_solve_scipy,
    print_results,
    check_docplex,
    check_cplex
)

print("✅ All functions imported successfully from reverse_logistics_cplex.py")
print()
print(f"DOcplex available : {check_docplex()}")
print(f"CPLEX engine      : {check_cplex()}")
print()
if check_docplex():
    print("🟢 Ready to solve with IBM CPLEX!")
else:
    print("🔴 DOcplex not found. Check Cell 1 output and reinstall.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Load Your ML Predictions
# This loads the predictions_2024_RL.csv file from your RF model
# ═══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

# ┌─────────────────────────────────────────────────────────┐
# │  ⚠️  UPDATE THIS PATH to your predictions CSV file      │
# └─────────────────────────────────────────────────────────┘
PREDICTIONS_CSV = r"C:\4-2\Thesis\data\Reverse logistics\predictions_2024_RL.csv"

df = pd.read_csv(PREDICTIONS_CSV)
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

print(f"✅ Loaded predictions: {len(df)} rows, {df.columns.tolist()[:6]}...")
print(f"   Date range: {df.index.min().date()} → {df.index.max().date()}")
print()

# Preview
cols_to_show = [
    'Solid_Waste_kg_Predicted',
    'Yarn_Waste_kg_Predicted', 
    'Chemical_Waste_kg_Predicted',
    'Treated_Wastewater_L'
]
print("Prediction columns preview:")
print(df[cols_to_show].head(5).to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Select a Date & Build Input Data
# Change TARGET_DATE to run optimization for any day in 2024
# ═══════════════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────────────┐
# │  Pick the date you want to optimize logistics for       │
# └─────────────────────────────────────────────────────────┘
TARGET_DATE = "2024-01-01"   # ← Change this to any date in your dataset

row = df.loc[TARGET_DATE]

# Pull predicted waste values from your ML model output
solid_kg    = float(row['Solid_Waste_kg_Predicted'])
yarn_kg     = float(row['Yarn_Waste_kg_Predicted'])
chemical_kg = float(row['Chemical_Waste_kg_Predicted'])
treated_ww  = float(row['Treated_Wastewater_L'])

print(f"📅 Target Date   : {TARGET_DATE}")
print(f"📦 Solid Waste   : {solid_kg:,.2f} kg  (Actual: {row['Solid_Waste_kg_Actual']:,.2f})")
print(f"🧵 Yarn Waste    : {yarn_kg:,.2f} kg  (Actual: {row['Yarn_Waste_kg_Actual']:,.2f})")
print(f"🧪 Chemical Waste: {chemical_kg:,.2f} kg  (Actual: {row['Chemical_Waste_kg_Actual']:,.2f})")
print(f"💧 Treated Water : {treated_ww:,.2f} L")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Define All MILP Parameters
# Edit these values to match your actual CC contracts
# ═══════════════════════════════════════════════════════════

data = {
    
    # ── Waste quantities (from Cell 4, auto-filled from ML predictions) ──────
    "wastes": {
        "solid":    solid_kg,
        "yarn":     yarn_kg,
        "chemical": chemical_kg,
    },
    "treated_ww": treated_ww,
    
    # ── Factory holding & preparation costs (BDT per kg) ────────────────────
    # Cost to store and prepare each waste type before dispatch
    "holding": {
        "solid":    2.5,   # BDT/kg
        "yarn":     3.0,   # BDT/kg  
        "chemical": 4.0,   # BDT/kg (higher due to hazardous handling)
    },
    
    # ── Transportation costs (BDT per kg per km) ─────────────────────────────
    "transport": {
        "solid":    0.05,  # BDT/kg/km
        "yarn":     0.04,  # BDT/kg/km
        "chemical": 0.08,  # BDT/kg/km (hazmat premium)
    },
    
    # ── ETP (Effluent Treatment Plant) parameters ────────────────────────────
    "etp": {
        "recycling_rate":    0.75,    # 75% of treated water is reused in-factory
        "reuse_value_per_L": 0.002,   # BDT saved per litre reused
        "drain_cost_per_L":  0.0005,  # BDT disposal cost per litre drained
    },
    
    # ── Working capital / time-value rate ────────────────────────────────────
    # Daily opportunity cost of capital tied up during CC processing
    # 0.0003 per day ≈ 10% annual rate / 365 days
    "wc_rate": 0.0003,
    
    # ── Environmental parameters ─────────────────────────────────────────────
    "co2_penalty":       1.5,   # BDT penalty per kg CO2-equivalent emitted
    "circularity_bonus": 2.0,   # BDT bonus per kg × circularity rate (ESG value)
    
    # ── Collection Centers (CCs) ─────────────────────────────────────────────
    # Add, remove, or edit CCs below. Each CC needs all 3 waste facilities.
    "ccs": [
        {
            "name": "GreenCycle BD (Gazipur)",
            "distance": 45,                  # km from factory
            "fixed_contract_cost": 5000,     # BDT — paid once when this CC is activated (binary y=1)
            "facilities": {
                "solid": {
                    "capacity":        2000,  # kg/day max this CC can process
                    "revenue_per_kg":  8.0,   # BDT they pay you per kg solid waste
                    "disposal_fee_per_kg": 0,
                    "process_days":    7,     # days until transaction completes → affects working capital cost
                    "co2_per_kg":      0.8,   # kg CO2-eq emitted per kg processed
                    "circularity_rate": 0.85, # fraction recycled back into circular economy
                },
                "yarn": {
                    "capacity":        1000,
                    "revenue_per_kg":  12.0,
                    "disposal_fee_per_kg": 0,
                    "process_days":    5,
                    "co2_per_kg":      0.5,
                    "circularity_rate": 0.90,
                },
                "chemical": {
                    "capacity":        4000,
                    "revenue_per_kg":  0,
                    "disposal_fee_per_kg": 5.0,  # BDT they CHARGE you per kg chemical sludge
                    "process_days":    10,
                    "co2_per_kg":      2.0,
                    "circularity_rate": 0.20,
                },
            }
        },
        {
            "name": "EcoTex Recyclers (Narayanganj)",
            "distance": 30,
            "fixed_contract_cost": 4000,
            "facilities": {
                "solid": {
                    "capacity":        1500,
                    "revenue_per_kg":  7.0,
                    "disposal_fee_per_kg": 0,
                    "process_days":    6,
                    "co2_per_kg":      0.9,
                    "circularity_rate": 0.80,
                },
                "yarn": {
                    "capacity":        800,
                    "revenue_per_kg":  11.5,
                    "disposal_fee_per_kg": 0,
                    "process_days":    4,
                    "co2_per_kg":      0.6,
                    "circularity_rate": 0.88,
                },
                "chemical": {
                    "capacity":        3500,
                    "revenue_per_kg":  0,
                    "disposal_fee_per_kg": 6.0,
                    "process_days":    12,
                    "co2_per_kg":      2.2,
                    "circularity_rate": 0.15,
                },
            }
        },
        {
            "name": "Circular Hub (Savar)",
            "distance": 20,
            "fixed_contract_cost": 3000,
            "facilities": {
                "solid": {
                    "capacity":        1000,
                    "revenue_per_kg":  6.5,
                    "disposal_fee_per_kg": 0,
                    "process_days":    8,
                    "co2_per_kg":      1.0,
                    "circularity_rate": 0.75,
                },
                "yarn": {
                    "capacity":        600,
                    "revenue_per_kg":  10.0,
                    "disposal_fee_per_kg": 0,
                    "process_days":    6,
                    "co2_per_kg":      0.7,
                    "circularity_rate": 0.82,
                },
                "chemical": {
                    "capacity":        2000,
                    "revenue_per_kg":  0,
                    "disposal_fee_per_kg": 4.5,
                    "process_days":    8,
                    "co2_per_kg":      1.8,
                    "circularity_rate": 0.25,
                },
            }
        },
    ]
}

print("✅ Data dictionary built successfully.")
print(f"   Wastes  : {data['wastes']}")
print(f"   CCs     : {[cc['name'] for cc in data['ccs']]}")
print(f"   ETP Rate: {data['etp']['recycling_rate']*100:.0f}% recycling")

In [ ]:
import sys, os, traceback

print('=' * 60)
print('DIAGNOSTIC REPORT')
print('=' * 60)

# 1. Python info
print('\n[1] Python version:', sys.version)
print('    Executable:', sys.executable)

# 2. Check docplex
print('\n[2] Testing: import docplex')
try:
    from docplex.mp.model import Model
    import docplex
    print('    OK - docplex version:', docplex.__version__)
except Exception as e:
    print('    FAILED:', e)
    print('    FIX: pip install docplex  (in thesis_cplex env)')

# 3. Check cplex engine
print('\n[3] Testing: cplex engine')
try:
    import cplex
    print('    OK - cplex version:', cplex.__version__)
except ImportError:
    print('    cplex standalone not found (checking via DOcplex next...)')

# 4. Test actual solve
print('\n[4] Testing: real CPLEX solve (small test problem)')
try:
    from docplex.mp.model import Model
    m = Model(name='test')
    x = m.continuous_var(name='x')
    m.maximize(x)
    m.add_constraint(x <= 10)
    sol = m.solve(log_output=False)
    if sol:
        val = sol.get_value(x)
        print('    OK - solver works, test result x =', val)
        print('    YOU ARE READY TO RUN CELL 6')
    else:
        print('    FAILED - solver returned no solution')
        print('    FIX: CPLEX runtime not linked to Python 3.10')
        print('    Open Anaconda Prompt and run:')
        print('      conda activate thesis_cplex')
        print('      cd "C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio2211\\cplex\\python\\3.10\\x64_win64"')
        print('      pip install .')
except Exception as e:
    print('    FAILED:', e)
    traceback.print_exc()
    print()
    print('    FIX: Run the following in Anaconda Prompt (thesis_cplex env):')
    print('      cd "C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio2211\\cplex\\python\\3.10\\x64_win64"')
    print('      pip install .')

# 5. CPLEX Studio environment variable
print('\n[5] CPLEX_STUDIO environment variable:')
studio = (os.environ.get('CPLEX_STUDIO_DIR2211') or
          os.environ.get('CPLEX_STUDIO_DIR201') or
          os.environ.get('CPLEX_STUDIO_DIR') or
          'NOT FOUND in environment')
print('   ', studio)

# 6. Check data dict
print('\n[6] Checking data dict from Cell 5...')
try:
    w = data['wastes']
    print('    OK - data dict is ready')
    print('    solid:', w['solid'], '  yarn:', w['yarn'], '  chemical:', w['chemical'])
except NameError:
    print('    MISSING - run Cells 3, 4, 5 first before this cell')

print('\n' + '=' * 60)
print('DIAGNOSTIC COMPLETE')
print('=' * 60)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Solve the MILP with IBM CPLEX
# ═══════════════════════════════════════════════════════════

print("🔄 Calling IBM CPLEX solver via DOcplex...")
print()

if check_docplex():
    res = build_and_solve_cplex(data)
    print("✅ CPLEX solve complete!")
else:
    print("⚠️  DOcplex not found — falling back to scipy solver (for testing only)")
    res = build_and_solve_scipy(data)
    print("✅ scipy solve complete (install docplex for IBM CPLEX)!")

print()
print(f"🏆 Optimal Z = {res['Z']:,.2f} BDT")
print(f"   Active CCs : {[data['ccs'][j]['name'] for j in res['active_ccs']]}")
print(f"   Allocations: {len(res['allocations'])} routes")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Print Full Optimal Solution
# ═══════════════════════════════════════════════════════════

print_results(res, data)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — View Results as a Clean DataFrame
# ═══════════════════════════════════════════════════════════

import pandas as pd

results_df = pd.DataFrame(res["allocations"])

# Round for display
display_cols = [
    'waste_type', 'cc_name', 'quantity', 'revenue', 'disposal_fee',
    'transport', 'holding', 'wc_cost', 'env_penalty', 
    'circ_bonus', 'net', 'process_days', 'circularity_rate', 'co2'
]
results_df_display = results_df[display_cols].copy()
for col in results_df_display.select_dtypes(include='float').columns:
    results_df_display[col] = results_df_display[col].round(2)

results_df_display.columns = [
    'Waste Type', 'CC Name', 'Qty (kg)', 'Revenue (BDT)', 'Disposal Fee (BDT)',
    'Transport (BDT)', 'Holding (BDT)', 'WC Cost (BDT)', 'Env Penalty (BDT)',
    'Circ Bonus (BDT)', 'Net (BDT)', 'Process Days', 'Circularity Rate', 'CO2 (kg)'
]

print(f"📊 Allocation Results for {TARGET_DATE}:")
print()
display(results_df_display)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Financial Summary Table
# ═══════════════════════════════════════════════════════════

allocs = res["allocations"]
ccs    = data["ccs"]

total_fixed = sum(ccs[j]["fixed_contract_cost"] for j in res["active_ccs"])

summary = {
    "Component": [
        "Revenue (Solid + Yarn)",
        "Disposal Fees (Chemical Sludge)",
        "Transportation Costs",
        "Holding & Preparation Costs",
        "Fixed Contract Costs  ★",
        "Working Capital Costs ★",
        "Environmental Penalties (CO2)",
        "Circularity Bonuses",
        "ETP Net Contribution",
        "──────────────────────────",
        "OPTIMAL Z (Objective Value)",
    ],
    "Amount (BDT)": [
        round(sum(a["revenue"]      for a in allocs), 2),
        round(-sum(a["disposal_fee"] for a in allocs), 2),
        round(-sum(a["transport"]   for a in allocs), 2),
        round(-sum(a["holding"]     for a in allocs), 2),
        round(-total_fixed, 2),
        round(-sum(a["wc_cost"]     for a in allocs), 2),
        round(-sum(a["env_penalty"] for a in allocs), 2),
        round(sum(a["circ_bonus"]   for a in allocs), 2),
        round(res["etp_net"], 2),
        "──────────",
        round(res["Z"], 2),
    ]
}

summary_df = pd.DataFrame(summary)
print(f"💰 Financial Summary — {TARGET_DATE}")
print()
display(summary_df.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Save Results to CSV and Excel
# ═══════════════════════════════════════════════════════════
import os
import subprocess
import sys

# Install openpyxl if missing
try:
    import openpyxl
except ModuleNotFoundError:
    print("openpyxl not found — installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])
    import openpyxl
    print("✅ openpyxl installed")

OUTPUT_FOLDER = PY_FILE_FOLDER
date_str = TARGET_DATE.replace("-", "")

csv_path   = os.path.join(OUTPUT_FOLDER, f"milp_allocation_{date_str}.csv")
excel_path = os.path.join(OUTPUT_FOLDER, f"milp_results_{date_str}.xlsx")

# Save CSV
results_df.to_csv(csv_path, index=False)
print(f"✅ CSV saved  : {csv_path}")

# Save Excel
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name="Allocation_Plan", index=False)

    summary_sheet = pd.DataFrame({
        "Component": [
            "Target Date", "Solver", "Optimal Z (BDT)",
            "Revenue (BDT)", "Disposal Fees (BDT)",
            "Transport (BDT)", "Holding (BDT)",
            "Fixed Contract (BDT)", "Working Capital (BDT)",
            "Env Penalty (BDT)", "Circ Bonus (BDT)",
            "ETP Net (BDT)", "Active CCs",
            "Total CO2 (kg)", "Avg Circularity Rate",
        ],
        "Value": [
            TARGET_DATE, "IBM CPLEX via DOcplex", round(res["Z"], 2),
            round(sum(a["revenue"] for a in allocs), 2),
            round(sum(a["disposal_fee"] for a in allocs), 2),
            round(sum(a["transport"] for a in allocs), 2),
            round(sum(a["holding"] for a in allocs), 2),
            round(sum(ccs[j]["fixed_contract_cost"] for j in res["active_ccs"]), 2),
            round(sum(a["wc_cost"] for a in allocs), 2),
            round(sum(a["env_penalty"] for a in allocs), 2),
            round(sum(a["circ_bonus"] for a in allocs), 2),
            round(res["etp_net"], 2),
            ", ".join(ccs[j]["name"] for j in res["active_ccs"]),
            round(sum(a["co2"] for a in allocs), 2),
            f"{sum(a['circularity_rate']*a['quantity'] for a in allocs)/sum(a['quantity'] for a in allocs)*100:.2f}%",
        ]
    })
    summary_sheet.to_excel(writer, sheet_name="Summary", index=False)

print(f"✅ Excel saved : {excel_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — (Optional) Run for Multiple Dates — Batch Mode
# This runs the MILP for every day in a date range and
# collects all Z values + allocations into one DataFrame
# ═══════════════════════════════════════════════════════════

from datetime import timedelta

BATCH_START = "2024-01-01"
BATCH_END   = "2024-01-07"   # ← Change range as needed (e.g. full year = "2024-12-31")

dates = pd.date_range(BATCH_START, BATCH_END)
batch_results = []

print(f"Running MILP for {len(dates)} dates: {BATCH_START} → {BATCH_END}")
print()

for i, date in enumerate(dates):
    date_str = date.strftime("%Y-%m-%d")
    
    if date_str not in df.index.strftime("%Y-%m-%d"):
        print(f"  ⚠ {date_str} not in predictions — skipping")
        continue
    
    row_b = df.loc[date_str]
    
    # Update waste quantities from predictions
    data_batch = dict(data)  # shallow copy
    data_batch["wastes"] = {
        "solid":    float(row_b['Solid_Waste_kg_Predicted']),
        "yarn":     float(row_b['Yarn_Waste_kg_Predicted']),
        "chemical": float(row_b['Chemical_Waste_kg_Predicted']),
    }
    data_batch["treated_ww"] = float(row_b['Treated_Wastewater_L'])
    data_batch["ccs"] = data["ccs"]       # reuse same CC config
    data_batch["holding"]  = data["holding"]
    data_batch["transport"] = data["transport"]
    data_batch["etp"] = data["etp"]
    data_batch["wc_rate"] = data["wc_rate"]
    data_batch["co2_penalty"] = data["co2_penalty"]
    data_batch["circularity_bonus"] = data["circularity_bonus"]
    
    try:
        if check_docplex():
            res_b = build_and_solve_cplex(data_batch)
        else:
            res_b = build_and_solve_scipy(data_batch)
        
        allocs_b = res_b["allocations"]
        batch_results.append({
            "Date":              date_str,
            "Z_Optimal (BDT)":  round(res_b["Z"], 2),
            "Solid_Pred (kg)":   data_batch["wastes"]["solid"],
            "Yarn_Pred (kg)":    data_batch["wastes"]["yarn"],
            "Chemical_Pred (kg)": data_batch["wastes"]["chemical"],
            "Revenue (BDT)":     round(sum(a["revenue"]      for a in allocs_b), 2),
            "Disposal (BDT)":    round(sum(a["disposal_fee"] for a in allocs_b), 2),
            "Transport (BDT)":   round(sum(a["transport"]    for a in allocs_b), 2),
            "WC_Cost (BDT)":     round(sum(a["wc_cost"]      for a in allocs_b), 2),
            "CO2_Total (kg)":    round(sum(a["co2"]          for a in allocs_b), 2),
            "Active_CCs":        len(res_b["active_ccs"]),
        })
        print(f"  ✅ {date_str}  Z = {res_b['Z']:>12,.2f} BDT")
        
    except Exception as e:
        print(f"  ❌ {date_str}  Error: {e}")

print()
batch_df = pd.DataFrame(batch_results)
print("📊 Batch Results:")
display(batch_df)

# Save batch results
batch_csv = os.path.join(OUTPUT_FOLDER, f"milp_batch_{BATCH_START[:7]}.csv")
batch_df.to_csv(batch_csv, index=False)
print(f"\n✅ Batch results saved: {batch_csv}")